# Evaluate Hopkins data

- Import cleaned raw data
- Transform on ML pipeline
- Pass through models

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.preprocess import remove_prefix
from src.data_utils import get_data, get_models
from src.eval import evaluate_models
import pandas as pd
import joblib
import numpy as np
import warnings


print(f"Path: {BASE_PATH}")

Globals

In [ ]:
# Data
DATA_DICT = {"base": get_data(is_nomo=False), "nomo": get_data(is_nomo=True)}
# Hopkins
HOPKINS_DICT = {
    "base": {
        "X": pd.read_parquet(
            BASE_PATH / "data" / "processed" / "hopkins" / "base" / "X.parquet"
        ),
        "y": pd.read_excel(
            BASE_PATH / "data" / "processed" / "hopkins" / "base" / "y.xlsx",
            index_col=0,
        ),
    },
    "nomo": {
        "X": pd.read_parquet(
            BASE_PATH / "data" / "processed" / "hopkins" / "nomo" / "X.parquet"
        ),
        "y": pd.read_excel(
            BASE_PATH / "data" / "processed" / "hopkins" / "nomo" / "y.xlsx",
            index_col=0,
        ),
    },
}


# Models
model_dir = BASE_PATH / "models" / "trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]
base_model_dict = {}

## Base models
base_model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
nomo_model_dict = get_models(["lr"], model_dir)

## Evaluate

- Use traditional test as val
- Use hopkins as test

In [ ]:
all_models_test_dict = {}
rows = []
index = []
n_bootstraps = 5000
save_path = BASE_PATH / "results"

Base

In [ ]:
## BASE MODELS
class_report_dict = evaluate_models(
    model_dict=base_model_dict,
    X_train=DATA_DICT["base"]["X_train"],
    y_train=DATA_DICT["base"]["y_train"].values.ravel(),
    X_val=DATA_DICT["base"]["X_test"],
    y_val=DATA_DICT["base"]["y_test"].values.ravel(),
    X_test=HOPKINS_DICT["base"]["X"],
    y_test=HOPKINS_DICT["base"]["y"].values.ravel(),
    results_path=save_path,
    threshold_str="val",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
## ONLY export test (have access to train/val if need be)
for model, metrics in class_report_dict["test"].items():
    rows.append(metrics)
    index.append(model)

Nomogram

In [ ]:
## BASE MODELS
class_report_dict = evaluate_models(
    model_dict=nomo_model_dict,
    X_train=DATA_DICT["nomo"]["X_train"],
    y_train=DATA_DICT["nomo"]["y_train"].values.ravel(),
    X_val=DATA_DICT["nomo"]["X_test"],
    y_val=DATA_DICT["nomo"]["y_test"].values.ravel(),
    X_test=HOPKINS_DICT["nomo"]["X"],
    y_test=HOPKINS_DICT["nomo"]["y"].values.ravel(),
    results_path=save_path,
    threshold_str="val",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
## ONLY export test (have access to train/val if need be)
for model, metrics in class_report_dict["test"].items():
    rows.append(metrics)
    index.append(model)

In [ ]:
all_models_outcomes_df = pd.DataFrame(rows, index=index)
report_path = save_path / "tables" / "class_report" / "class_report.csv"
if report_path.exists():
    warnings.warn(f"Over-writing class report table at path: {report_path}")
    report_path.unlink()
report_path.parent.mkdir(exist_ok=True, parents=True)
all_models_outcomes_df.to_csv(report_path)